In [1]:
import sys

sys.path.append("..")
import numpy as np
import os
import pandas as pd
from pathlib import Path
from classes.sample.raw_data import INCFile, LMSFile
from pydantic import BaseModel
from classes.sample.sample_types import SampleTypes
from classes.sample.sample_classes import SampleFRF, SingleFRFSample
from classes.signal.signal_classes import SignalFRF
import json

In [2]:
path = Path(
    os.getenv("RAW_DATA_PATH", "data/raw")
)

In [3]:
output_path = Path(r"..\\data\\frfs\\sequence_1\\")

In [4]:
folders = os.listdir(path)

In [ ]:
from scipy.signal import welch, csd
from typing import Tuple

def get_frfs_from_lms(lms_file: LMSFile, nperseg: int) -> Tuple[SampleFRF, SampleFRF]:
    """
    Compute the frequency response functions (FRFs) H1 and H2 from LMS data.

    Args:
        lms_file (LMSFile): An instance of LMSFile containing the input force and response data.
        nperseg (int): Length of each segment for the Welch method.

    Returns:
        Tuple[SampleFRF, SampleFRF]: Two SampleFRF instances containing the H1 and H2 FRFs for each cable and axis.
    """
    f_input = lms_file.input_force
    dt = lms_file.dt
    fs = 1 / dt
    h1_per_cable = {}
    h2_per_cable = {}

    for cable in [1, 2, 3, 4]:
        h1_per_axis = {}
        h2_per_axis = {}
        tension = float(np.mean(getattr(lms_file, f"s{cable}")))

        for axis in ["x", "y", "z"]:
            x = getattr(lms_file, f"c{cable}_{axis}")
            f, xx = welch(x=f_input, fs=fs, window="hann", nperseg=nperseg)
            _, yy = welch(x=x, fs=fs, window="hann", nperseg=nperseg)
            _, xy = csd(x=x, y=f_input, fs=fs, nperseg=nperseg, window="hamming")

            h1 = xy / xx
            h2 = yy / xy

            h1_per_axis[axis] = SignalFRF(
                frequencies=f.tolist(),
                abs=np.abs(h1).tolist(),
                angle=np.angle(h1).tolist(),
            )
            h2_per_axis[axis] = SignalFRF(
                frequencies=f.tolist(),
                abs=np.abs(h2).tolist(),
                angle=np.angle(h2).tolist(),
            )

        h1_per_cable[cable] = SingleFRFSample(
            x=h1_per_axis["x"],
            y=h1_per_axis["y"],
            z=h1_per_axis["z"],
            tension=tension,
        )
        h2_per_cable[cable] = SingleFRFSample(
            x=h2_per_axis["x"],
            y=h2_per_axis["y"],
            z=h2_per_axis["z"],
            tension=tension,
        )

    return SampleFRF(
        sample_id=lms_file.sampled_at.isoformat(),
        sample_c1=h1_per_cable[1],
        sample_c2=h1_per_cable[2],
        sample_c3=h1_per_cable[3],
        sample_c4=h1_per_cable[4],
    ), SampleFRF(
        sample_id=lms_file.sampled_at.isoformat(),
        sample_c1=h2_per_cable[1],
        sample_c2=h2_per_cable[2],
        sample_c3=h2_per_cable[3],
        sample_c4=h2_per_cable[4],
    )

# FRF

In [ ]:
processed_docs = []
count = 0
for folder in folders:
    if not folder.startswith("Teste") or folder.endswith(".npy"):
        continue
    files = os.listdir(path / folder)
    prefixes = [f[:-8] for f in files if f.endswith(".txt")]
    prefixes = np.unique(prefixes)
    for prefix in prefixes:
        if prefix in processed_docs:
            continue
        #if count >= 1:
        #    break
        processed_docs.append(prefix)
        lms_path = path / folder / f"{prefix}_LMS.txt"
        try:
            lms_file = LMSFile.from_txt(lms_path)
            # Compute FRF
            sample_h1, sample_h2 = get_frfs_from_lms(lms_file=lms_file, nperseg=2048)
            count += 1
            
            # Save to JSON
            filename = sample_h1.sample_id + ".json"
            filename = filename.replace(":", "-")
            with open(output_path / filename, "w+") as h:
                json.dump(sample_h1.model_dump(), h)        
        except:
           print(f"Fail: {lms_path}")    

In [6]:
output_path

WindowsPath('../data/sequence_1_frf')